## Отчет по лабораторной работе №1
## «Линейная регрессия и факторный анализ»


## Введение

Цель работы: изучение основ линейной регрессии, построение простейших моделей регрессии, проведение обучения модели на реальных данных и оценка её качество.

Постановка задачи:
1) Провести обучение модели линейной регрессии на датасете с Kaggle:
2) Загрузить датасет из репозитория (например, kaggle.com или аналогичных платформ).
3) Подготовить данные: провести первичный анализ, визуализировать распределение признаков и целевой переменной. 
4) Провести предобработку данных: удалить пропущенные значения, закодировать категориальные переменные (опционально).
5) Построить матрицу корреляций. Сделать выводы о наличии мультиколлинеарности (расчет VIF-коэффициента).
6) Построить регрессионные модели (линейная, лассо и гребневая). Если целевая переменная - категориальная, то исследовать логистическую регрессию. Разделить на тренировочную и тестовую выборки (80/20 или 70/30). Использовать кросс-валидацию. Оценить качество построенной модели с помощью метрик: RMSE (Root Mean Square Error), R² (коэффициент детерминации) и MAPE (Mean Absolute Percentage Error).
7) Построить график остатков моделей (добавить горизонтальную линию на уровне 0) и исследовать структуру остатков на предмет гомоскедастичности или гетероскедастичности.
8) Устранить мультиколлинеарность и снизить размерность признаков с помощью метода главных компонент (PCA). Перед проведением PCA провести стандартизацию данных.
9) Повторить шаг 5 (линейная, лассо и гребневая регрессия), но использовать в качестве признаков не исходные данные, а главные компоненты. Сравнить метрики качества (RMSE, R² и MAPE) моделей, обученных на исходных данных и на главных компонентах.


## Описание датасета.
Датасет описывает данные о продаже недвижимости в США. 

* цена продажи (price) — количественный вещественный (целевая переменная);
* количество спален (bedrooms) — количественный дискретный;
* количество ванных комнат (bathrooms) — количественный вещественный;
* жилая площадь (sqft_living) — количественный дискретный (в квадратных футах);
* площадь участка (sqft_lot) — количественный дискретный (в квадратных футах);
* количество этажей (floors) — количественный вещественный;
* выход к воде (waterfront) — категориальный бинарный;
* оценка вида (view) — категориальный порядковый (от 0 до 4);
* состояние дома (condition) — категориальный порядковый (от 1 до 5);
* надземная площадь (sqft_above) — количественный дискретный (в квадратных футах);
* площадь подвала (sqft_basement) — количественный дискретный (в квадратных футах);
* год постройки (yr_built) — количественный дискретный;
* год реновации (yr_renovated) — количественный дискретный;


## Первичный анализ

Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, LassoCV, Ridge, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.decomposition import PCA

In [ ]:
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13


df = pd.read_csv('ДанныеДляАнализа.csv', sep=';')

print("Первичный анализ")
print(f"Размер датасета: {df.shape}")
print(f"Количество пропусков: {df.isna().sum().sum()}")
print(f"Количество дубликатов: {df.duplicated().sum()}")

# Описательная статистика

In [ ]:
print("\nОписательная статистика:")
print(df.describe())

# Выделение числовых признаков и целевой переменной, а так же удаление дубликатов и пропусков

In [ ]:
num_cols = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 
            'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 
            'yr_built', 'yr_renovated']
target = 'price'

X = df[num_cols].copy()
y = df[target].copy()


df_clean = df.dropna().drop_duplicates().reset_index(drop=True)
X = df_clean[num_cols]
y = df_clean[target]


# Визуализация распределений числовых признаков и целевой переменной

In [ ]:
axes = X[num_cols].hist(figsize=(16, 12), bins=30,
                        edgecolor='black', color='steelblue')
for ax, col in zip(axes.flatten(), num_cols):
    ax.set_xlabel(col)
    ax.set_ylabel("Частота")
    ax.set_title(col)
plt.suptitle("Распределение числовых признаков", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()


fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(y=X[col], ax=axes[i], color='steelblue')
    axes[i].set_title(col)
for j in range(len(num_cols), len(axes)):
    axes[j].axis('off')
plt.suptitle("Boxplot числовых признаков", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))
sns.histplot(y, bins=50, kde=True, color='coral')
plt.title("Распределение целевой переменной (price)")
plt.xlabel("Цена")
plt.ylabel("Частота")
plt.tight_layout()
plt.show()

print("\nОписательная статистика целевой переменной:")
print(y.describe())
print("Асимметрия:", y.skew().round(2))



key_features = ['sqft_living', 'sqft_lot', 'sqft_above', 'sqft_basement',
                'bedrooms', 'bathrooms', 'floors', 'yr_built']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(key_features):
    sns.scatterplot(data=X, x=col, y=y, alpha=0.3, ax=axes[i], color='coral')
    axes[i].set_title(f"{col} vs price")
    axes[i].set_ylim(0, y.quantile(0.99))

plt.tight_layout()
plt.show()



discrete_features = ['bedrooms', 'bathrooms', 'floors',
                     'waterfront', 'view', 'condition']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(discrete_features):
    sns.boxplot(data=X, x=col, y=y, ax=axes[i], color='coral')
    axes[i].set_title(f"price vs {col}")
    axes[i].set_ylim(0, y.quantile(0.99))

plt.tight_layout()
plt.show()

Анализ распределения числовых признаков
Построены гистограммы распределения всех числовых признаков:
* sqft_living (жилая площадь) — выраженная правосторонняя асимметрия. Большинство домов имеют площадь от 1200 до 2500 кв. футов, пик в районе 1500–2000 кв. футов. Есть длинный хвост вправо с элитными особняками до 13 540 кв. футов.
* sqft_lot (площадь участка) — экстремальная правосторонняя асимметрия. Подавляющее большинство участков имеют площадь до 10 000 кв. футов, но единичные выбросы достигают более 1 млн кв. футов.
* bedrooms (количество спален) — дискретный признак: основная масса домов имеет 3 или 4 спальни (более 1100 объектов из 1500). Редкие значения (1, 6–9 спален) единичны.
* bathrooms (количество ванных комнат) — распределение скошено вправо с пиками около значений 1.75, 2.0 и 2.5.
* floors (этажность) — дискретный/порядковый признак: доминируют одноэтажные (1.0) и двухэтажные (2.0) дома, также представлены полутораэтажные (1.5) объекты.
* yr_built (год постройки) — распределение относительно равномерное с некоторым преобладанием более современных построек (вторая половина XX века и начало XXI века).
* sqft_above и sqft_basement — надземная площадь повторяет форму распределения общей площади, а подвальная площадь имеет большой пик в нуле (у множества домов подвал отсутствует).

Общие выводы:
* Признаки имеют существенно различающиеся масштабы (например, sqft_lot исчисляется сотнями тысяч, а floors или waterfront — единицами), что строго обосновывает необходимость обязательной стандартизации перед проведением PCA и применением регуляризованных моделей.
* Ряд признаков (sqft_living, sqft_lot) имеют выраженную правостороннюю асимметрию и тяжелые хвосты выбросов.




Анализ выбросов (boxplot)
Построены boxplot'ы для всех числовых признаков.
* sqft_living — медиана около 1935 кв. футов, межквартильный разброс от 1410 до 2550. За верхней границей усов лежит значительное количество выбросов (крупные дома от 4500 до 13 540 кв. футов). Это реальные объекты премиум-класса, а не ошибки ввода.
* sqft_lot — из-за гигантских загородных участков шкала сильно сжата, большинство точек сосредоточено около нуля, а выбросы тянутся до 1.07 млн кв. футов.
* bedrooms / bathrooms — выбросы фиксируют объекты с большим количеством комнат (до 9 спален и 8 ванных), представляющие собой элитную недвижимость.
* yr_built — выбросов снизу нет, минимальный год постройки в выборке — 1900.

Общие выводы:
* Выбросы в признаках площади и комнатности являются экономически реальными (объекты элитного сегмента).
* Наличие таких выбросов приводит к смещению линейных моделей, что компенсируется стандартизацией и логарифмированием целевой переменной.


Анализ целевой переменной price
Распределение целевой переменной сильно асимметрично (выраженная правосторонняя асимметрия):
* Статистика:
* Среднее: $540 111
* Медиана: $465 000
* Стандартное отклонение: $326 890
* Минимум: $80 000
* Максимум: $3 200 000
* 25-й процентиль: $326 400
* 75-й процентиль: $642 000
* Асимметрия: 2.40 (высокая положительная асимметрия).

* Наблюдения:
* Среднее превышает медиану почти на $75 000, что характерно для цен на недвижимость: дорогие особняки «тянут» среднее значение вверх.
* Диапазон цен различается в 40 раз (от $80 тыс. до $3.2 млн).
* 75% домов стоят дешевле $642 000, но максимальная цена достигает $3.2 млн.
* Экстремальная асимметрия (2.40) подтверждает правильность решения о логарифмировании целевой переменной ($\ln(1 + \text{price})$) для стабилизации дисперсии остатков в регрессионных моделях.


 Анализ категориальных и бинарных признаков
* waterfront (выход к воде) — крайне несбалансированный бинарный признак. Доля домов с выходом к воде составляет менее 1% (около 6 объектов из 1500).
* view (оценка вида) — порядковый признак (от 0 до 4). Большинство объектов имеют оценку 0 (оценка отсутствует), категории 1–4 встречаются значительно реже.
* condition (состояние дома) — порядковый признак (от 1 до 5). Преобладают дома с оценкой состояния 3 и 4 (в сумме более 85% выборки). Оценки 1 и 2 единичны.

Общие выводы: 

* Бинарные и порядковые признаки имеют сильный дисбаланс классов (например, waterfront).
* Такие признаки несут специфическую информацию, но из-за сильного перекоса требуют аккуратного обращения при построении линейных моделей.

Связь признаков с целевой переменной
Построены диаграммы рассеяния и boxplot'ы для ключевых признаков относительно цены (price):
* sqft_living vs price: Четкая положительная связь. Цены закономерно растут с увеличением жилой площади. Разброс цен увеличивается для больших домов (проявление гетероскедастичности). Это сильнейший линейный предиктор.
* sqft_lot vs price: Слабая связь. Размер участка слабо влияет на итоговую стоимость дома в данной выборке, преобладают плотные облака точек в нижней части графика.
* bedrooms / bathrooms vs price: Положительная тенденция — медианная цена дома ступеньками растет по мере увеличения количества спален и ванных комнат.
* waterfront vs price: Дома с выходом к воде (waterfront = 1) имеют медианную цену на $600–700 тыс. выше стандартных объектов, хотя таких наблюдений немного.
* view vs price: С ростом оценки вида (от 0 до 4) медианная стоимость недвижимости существенно возрастает.

Jбщие выводы:
* Главными драйверами цены являются жилая площадь (sqft_living), надземная площадь (sqft_above), а также качественные характеристики (view, waterfront).
* Наблюдается гетероскедастичность (увеличение разброса остатков и цен для дорогих объектов), что подтверждает необходимость логарифмирования.

# Построение корелляционной матрицы

In [ ]:
plt.figure(figsize=(12, 10))
corr = df[num_cols + [target]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Корреляционная матрица признаков и цены")
plt.tight_layout()
plt.show()

# VIF коэффициент

In [ ]:
X_vif = X.drop(columns=['sqft_living']) 
vif_data = pd.DataFrame()
vif_data["Признак"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print("\nКоэффициенты VIF ")
print(vif_data.sort_values(by="VIF", ascending=False).to_string(index=False))

# Анализ результатов

Согласно результатам оценки по VIF-коэффициенту, мультиколлинеарность есть, так как есть значение VIF у признака bathrooms равное 3.22.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=num_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=num_cols)

#  Обучение моделей

1. Линейная регрессия

In [ ]:
model_lr = LinearRegression()

cv_r2 = cross_val_score(
    model_lr, X_train_scaled, y_train, cv=5, scoring='r2'
)
print('Кросс-валидация Linear Regression (R²):')
print(f'  Фолды:   {cv_r2.round(4)}')
print(f'  Среднее: {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')

model_lr.fit(X_train_scaled, y_train)
y_pred = model_lr.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print('\nМетрики Linear Regression на тесте (без логарифма):')
print(f'  RMSE: {rmse:,.0f} руб.   (в среднем модель ошибается на эту сумму)')
print(f'  R²:   {r2:.4f}            (модель объясняет {r2*100:.1f}% вариации цены)')
print(f'  MAPE: {mape*100:.2f}%          (средняя ошибка в процентах)')



y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

model_lr_log = LinearRegression()

cv_r2_log = cross_val_score(
    model_lr_log, X_train_scaled, y_train_log, cv=5, scoring='r2'
)
print('\n\nКросс-валидация Linear Regression с логарифмом (R²):')
print(f'  Фолды:   {cv_r2_log.round(4)}')
print(f'  Среднее: {cv_r2_log.mean():.4f} ± {cv_r2_log.std():.4f}')

model_lr_log.fit(X_train_scaled, y_train_log)
y_pred_log = model_lr_log.predict(X_test_scaled)


y_pred_log_orig = np.expm1(y_pred_log)
y_test_orig = np.expm1(y_test_log)  

rmse_log = np.sqrt(mean_squared_error(y_test_orig, y_pred_log_orig))
r2_log = r2_score(y_test_orig, y_pred_log_orig)
mape_log = mean_absolute_percentage_error(y_test_orig, y_pred_log_orig)

print('\nМетрики Linear Regression на тесте (с логарифмом, в исходной шкале):')
print(f'  RMSE: {rmse_log:,.0f} руб.')
print(f'  R²:   {r2_log:.4f}  (объяснено {r2_log*100:.1f}%)')
print(f'  MAPE: {mape_log*100:.2f}%')


2. Lasso-регрессия

In [ ]:
alphas_lasso = np.logspace(-4, 1, 50)
lasso_cv = LassoCV(
    alphas=alphas_lasso, cv=5, random_state=42, max_iter=10000
)
lasso_cv.fit(X_train_scaled, y_train_log)
print(f'\n\nОптимальный alpha для Lasso: {lasso_cv.alpha_:.6f}')
print(
    f'Признаков с ненулевым коэффициентом: {(lasso_cv.coef_ != 0).sum()} из {len(lasso_cv.coef_)}'
)

model_lasso = Lasso(alpha=lasso_cv.alpha_, max_iter=10000)

cv_r2_lasso = cross_val_score(
    model_lasso, X_train_scaled, y_train_log, cv=5, scoring='r2'
)
print(f'CV R² Lasso: {cv_r2_lasso.mean():.4f} ± {cv_r2_lasso.std():.4f}')

model_lasso.fit(X_train_scaled, y_train_log)
y_pred_lasso = np.expm1(model_lasso.predict(X_test_scaled))

rmse_l = np.sqrt(mean_squared_error(y_test_orig, y_pred_lasso))
r2_l = r2_score(y_test_orig, y_pred_lasso)
mape_l = mean_absolute_percentage_error(y_test_orig, y_pred_lasso)

print('\nМетрики Lasso на тесте:')
print(f'  RMSE: {rmse_l:,.0f} руб.')
print(f'  R²:   {r2_l:.4f}')
print(f'  MAPE: {mape_l*100:.2f}%')


coef_df = pd.DataFrame(
    {'Признак': X_train_scaled.columns, 'Коэффициент': model_lasso.coef_}
).sort_values('Коэффициент', key=abs, ascending=False)
print('\nКоэффициенты признаков в Lasso-регрессии:')
print(coef_df.to_string(index=False))


3. Ridge-регрессия

In [ ]:
alphas_ridge = np.logspace(-4, 2, 50)
ridge_cv = RidgeCV(alphas=alphas_ridge, cv=5, scoring='r2')
ridge_cv.fit(X_train_scaled, y_train_log)
print(f'\n\nОптимальный alpha для Ridge: {ridge_cv.alpha_:.6f}')

model_ridge = Ridge(alpha=ridge_cv.alpha_)

cv_r2_ridge = cross_val_score(
    model_ridge, X_train_scaled, y_train_log, cv=5, scoring='r2'
)
print(f'CV R² Ridge: {cv_r2_ridge.mean():.4f} ± {cv_r2_ridge.std():.4f}')

model_ridge.fit(X_train_scaled, y_train_log)
y_pred_ridge = np.expm1(model_ridge.predict(X_test_scaled))

rmse_r = np.sqrt(mean_squared_error(y_test_orig, y_pred_ridge))
r2_r = r2_score(y_test_orig, y_pred_ridge)
mape_r = mean_absolute_percentage_error(y_test_orig, y_pred_ridge)

print('\nМетрики Ridge на тесте:')
print(f'  RMSE: {rmse_r:,.0f} руб.')
print(f'  R²:   {r2_r:.4f}')
print(f'  MAPE: {mape_r*100:.2f}%')

Сводная таблица

In [ ]:
comparison = pd.DataFrame(
    {
        'Модель': [
            'Linear (без логарифма)',
            'Linear (с логарифмом)',
            'Lasso',
            'Ridge',
        ],
        'CV R² (train)': [
            cv_r2.mean(),
            cv_r2_log.mean(),
            cv_r2_lasso.mean(),
            cv_r2_ridge.mean(),
        ],
        'RMSE (test)': [rmse, rmse_log, rmse_l, rmse_r],
        'R² (test)': [r2, r2_log, r2_l, r2_r],
        'MAPE (test)': [mape * 100, mape_log * 100, mape_l * 100, mape_r * 100],
    }
)

styled = (
    comparison.style.format({
        'CV R² (train)': '{:.4f}',
        'RMSE (test)': '{:,.0f}',
        'R² (test)': '{:.4f}',
        'MAPE (test)': '{:.2f}%',
    })
    .highlight_min(subset=['RMSE (test)', 'MAPE (test)'], color='green')
    .highlight_max(
        subset=['CV R² (train)', 'R² (test)'], color='green'
    )
    .set_caption(
        'Сравнение регрессионных моделей (до и после логарифмирования)'
    )
)

styled

# График остатков

In [ ]:
y_pred_log_model = model_lr_log.predict(X_test_scaled)
y_pred_orig = np.expm1(
    y_pred_log_model
) 
residuals = y_test - y_pred_orig 

# Строим график остатков
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_orig, residuals, alpha=0.4, color='royalblue', edgecolors='k', linewidth=0.5)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Уровень 0')

plt.xlabel('Предсказанные значения цены', fontsize=12)
plt.ylabel('Остатки (Факт - Прогноз)', fontsize=12)
plt.title('График остатков модели Линейной регрессии', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

Анализ остатков
На построенном графике остатков модели линейной регрессии точки в целом симметрично сгруппированы вокруг нулевой линии, что говорит об отсутствии общего системного смещения прогнозов. Анализ структуры распределения указывает на явное проявление гетероскедастичности: с ростом предсказанных значений вертикальный разброс остатков заметно увеличивается, образуя характерный расширяющийся вправо диапазон ошибок. Для объектов с низкой и средней ценой прогнозирование выполняется более кучно и стабильно, в то время как на сегменте дорогой недвижимости амплитуда ошибок возрастает, что свойственно рыночным данным о жилье и объясняется высокой вариативностью цен на элитные объекты.

# Анализ методом главных компонент

In [ ]:
pca = PCA()
X_train_pca_full = pca.fit_transform(X_train_scaled)
explained = pca.explained_variance_ratio_
cum_explained = np.cumsum(explained) * 100



fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(explained)+1), explained*100, color='steelblue', alpha=0.8)
axes[0].plot(range(1, len(explained)+1), explained*100, 'o-', color='coral', linewidth=2)
axes[0].set_xlabel("Номер главной компоненты")
axes[0].set_ylabel("Объясненная дисперсия, %")
axes[0].set_title("Scree plot — вклад каждой компоненты")
axes[0].grid(True, linestyle='--', alpha=0.5)


axes[1].plot(range(1, len(cum_explained)+1), cum_explained, 'o-', color='seagreen', linewidth=2)
axes[1].axhline(y=95, color='red', linestyle='--', label='95%')
axes[1].axhline(y=90, color='goldenrod', linestyle='--', label='90%')
axes[1].set_xlabel("Номер главной компоненты")
axes[1].set_ylabel("Накопленная объясненная дисперсия, %")
axes[1].set_title("Кумулятивная дисперсия")
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


n_components = np.argmax(cum_explained >= 95) + 1
print(f"\nКоличество компонент для 95% дисперсии: {n_components}")

pca_final = PCA(n_components=n_components)
X_train_pca = pca_final.fit_transform(X_train_scaled)
X_test_pca = pca_final.transform(X_test_scaled)

# Обучение моделей на главных компонентах PCA

In [ ]:
model_lr_pca = LinearRegression()
cv_r2_lr_pca = cross_val_score(
    model_lr_pca, X_train_pca, y_train_log, cv=5, scoring='r2'
)
model_lr_pca.fit(X_train_pca, y_train_log)
y_pred_lr_pca = np.expm1(model_lr_pca.predict(X_test_pca))

rmse_lr_pca = np.sqrt(mean_squared_error(y_test_orig, y_pred_lr_pca))
r2_lr_pca = r2_score(y_test_orig, y_pred_lr_pca)
mape_lr_pca = mean_absolute_percentage_error(y_test_orig, y_pred_lr_pca)

print('Linear Regression на PCA')
print(f'CV R²: {cv_r2_lr_pca.mean():.4f} ± {cv_r2_lr_pca.std():.4f}')
print(f'RMSE:  {rmse_lr_pca:,.0f} руб.')
print(f'R²:    {r2_lr_pca:.4f}')
print(f'MAPE:  {mape_lr_pca*100:.2f}%')



alphas = np.logspace(-4, 2, 50)
ridge_cv_pca = RidgeCV(alphas=alphas, cv=5, scoring='r2')
ridge_cv_pca.fit(X_train_pca, y_train_log)

model_ridge_pca = Ridge(alpha=ridge_cv_pca.alpha_)
cv_r2_ridge_pca = cross_val_score(
    model_ridge_pca, X_train_pca, y_train_log, cv=5, scoring='r2'
)
model_ridge_pca.fit(X_train_pca, y_train_log)
y_pred_ridge_pca = np.expm1(model_ridge_pca.predict(X_test_pca))

rmse_ridge_pca = np.sqrt(mean_squared_error(y_test_orig, y_pred_ridge_pca))
r2_ridge_pca = r2_score(y_test_orig, y_pred_ridge_pca)
mape_ridge_pca = mean_absolute_percentage_error(y_test_orig, y_pred_ridge_pca)

print('\nRidge Regression на PCA')
print(f'alpha: {ridge_cv_pca.alpha_:.4f}')
print(f'CV R²: {cv_r2_ridge_pca.mean():.4f} ± {cv_r2_ridge_pca.std():.4f}')
print(f'RMSE:  {rmse_ridge_pca:,.0f} руб.')
print(f'R²:    {r2_ridge_pca:.4f}')
print(f'MAPE:  {mape_ridge_pca*100:.2f}%')



lasso_cv_pca = LassoCV(
    alphas=np.logspace(-4, 1, 50), cv=5, random_state=42, max_iter=10000
)
lasso_cv_pca.fit(X_train_pca, y_train_log)

model_lasso_pca = Lasso(alpha=lasso_cv_pca.alpha_, max_iter=10000)
cv_r2_lasso_pca = cross_val_score(
    model_lasso_pca, X_train_pca, y_train_log, cv=5, scoring='r2'
)
model_lasso_pca.fit(X_train_pca, y_train_log)
y_pred_lasso_pca = np.expm1(model_lasso_pca.predict(X_test_pca))

rmse_lasso_pca = np.sqrt(mean_squared_error(y_test_orig, y_pred_lasso_pca))
r2_lasso_pca = r2_score(y_test_orig, y_pred_lasso_pca)
mape_lasso_pca = mean_absolute_percentage_error(y_test_orig, y_pred_lasso_pca)

print('\n Lasso Regression на PCA ')
print(f'alpha: {lasso_cv_pca.alpha_:.6f}')
print(f'CV R²: {cv_r2_lasso_pca.mean():.4f} ± {cv_r2_lasso_pca.std():.4f}')
print(f'RMSE:  {rmse_lasso_pca:,.0f} руб.')
print(f'R²:    {r2_lasso_pca:.4f}')
print(f'MAPE:  {mape_lasso_pca*100:.2f}%')

Сравнительная таблица до и после PCA

In [ ]:
comparison_pca = pd.DataFrame(
    {
        'Модель': ['Linear Regression', 'Ridge Regression', 'Lasso Regression'],
        'RMSE до PCA': [rmse_log, rmse_r, rmse_l],
        'RMSE после PCA': [rmse_lr_pca, rmse_ridge_pca, rmse_lasso_pca],
        'R² до PCA': [r2_log, r2_r, r2_l],
        'R² после PCA': [r2_lr_pca, r2_ridge_pca, r2_lasso_pca],
        'MAPE до PCA (%)': [
            mape_log * 100,
            mape_r * 100,
            mape_l * 100,
        ],
        'MAPE после PCA (%)': [
            mape_lr_pca * 100,
            mape_ridge_pca * 100,
            mape_lasso_pca * 100,
        ],
    }
)

comp_formatted = comparison_pca.copy()
comp_formatted['RMSE до PCA'] = comp_formatted['RMSE до PCA'].map('{:,.0f}'.format)
comp_formatted['RMSE после PCA'] = comp_formatted['RMSE после PCA'].map(
    '{:,.0f}'.format
)
comp_formatted['R² до PCA'] = comp_formatted['R² до PCA'].round(4)
comp_formatted['R² после PCA'] = comp_formatted['R² после PCA'].round(4)
comp_formatted['MAPE до PCA (%)'] = comp_formatted['MAPE до PCA (%)'].map(
    '{:.2f}%'.format
)
comp_formatted['MAPE после PCA (%)'] = comp_formatted['MAPE после PCA (%)'].map(
    '{:.2f}%'.format
)

print('Сравнение моделей: Исходные данные vs Главные компоненты (PCA)')
print(comp_formatted.to_string(index=False))

Анализ
Сравнение результатов моделей на исходных признаках и главных компонентах показало, что качество прогнозирования практически не изменилось: метрики RMSE, R^2 и MAPE остались на прежнем уровне для всех трех алгоритмов (линейной, гребневой и лассо регрессии). Применение метода главных компонент позволило успешно снизить размерность пространства признаков и устранить мультиколлинеарность без потери информационной емкости датасета, так как модели, обученные на ортогональных компонентах, продемонстрировали идентичную точность по сравнению с полным набором исходных данных.

## Заключение. Сделайте вывод о результатах работы.
Перед началом выполнения лабораторной работы был изучен датасет с предсказанием цены на недвижимость на основе ключевых признаков. Всего в датасете было 1500 строк и 18 столбцов. После проверки на дубликаты и пустые значения было выявлено, что пропусков и дубликатов в данных нет (осталось 1500 строк). Для анализа были отобраны 12 ключевых числовых признаков и целевая переменная (цена). Была выведена корреляционная матрица и рассчитаны коэффициенты VIF. У поля bathrooms значение VIF составило 3.22, что указывает на наличие умеренной мультиколлинеарности.
Подготовка данных перед обучением моделей происходила в следующем порядке: данные были разделены на тренировочную и тестовую выборки в соотношении 80/20, целевая переменная была прологарифмирована для стабилизации дисперсии, а независимые признаки были стандартизированы с помощью StandardScaler.
В ходе лабораторной работы было изучено построение регрессионных моделей предсказания стоимости недвижимости на основе заданных признаков. Были построены три разные модели: линейная регрессия, гребневая (Ridge) и лассо (Lasso) регрессия с подбором гиперпараметров через кросс-валидацию. По итогам сравнения показателей эффективности на исходных данных можно сделать вывод, что все три модели демонстрируют практически идентичные результаты (R^2≈0.59, MAPE ≈28%). Модель Lasso показала минимально лучший результат по R^2 (0.5912), однако разница в метриках между моделями крайне мала (менее 0.3%), что говорит об их одинаковой эффективности на данном наборе признаков. При анализе остатков модели линейной регрессии была обнаружена гетероскедастичность (увеличение разброса ошибок при росте предсказанных значений цен).
После проведения метода главных компонент (PCA) и анализа графика накопленной дисперсии было определено оптимальное количество компонент, равное 10 (для сохранения 95% дисперсии данных). Модели были заново обучены на главных компонентах вместо исходных признаков. Сравнение метрик показало, что качество моделей практически не изменилось: показатели RMSE, R^2 и MAPE на главных компонентах совпали с результатами на исходных данных с точностью до тысячных долей (разница в R^2 составила менее 0.001). Это доказывает успешность снижения размерности пространства признаков и полного устранения мультиколлинеарности без потери информационной ценности датасета.


## Список источников. Приведите использованную литературу и ссылки на ресурсы.
1. Датасет "House Price Prediction" [Электронный ресурс]. - URL: https://www.kaggle.com/datasets/debayank2024/house-price-prediction/data (дата обращения 13.09.2026).
2.  Kaggle : официальный сайт платформы для соревнований по машинному обучению. — URL: https://www.kaggle.com (дата обращения: 13.10.2026).
3. Электронный курс "Системы искусственного интеллекта и машинное обучение" в образовательной среде НГТУ НЭТИ Dispace 3.0. Постановка задачи к лабораторной работе №1 [Электронный ресурс]. - 2025. - URL: https://dispace.nstu.ru/course/965/9020/23586 (дата обращения 13.10.2026).